In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# JAX House Prices Model

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, jit
from jax.lax import fori_loop
import pandas as pd
import time   

# -------------------------
# Load Dataset
# -------------------------
path = "/kaggle/input/competitions/house-prices-advanced-regression-techniques/"
train = pd.read_csv(path + "train.csv")

# -------------------------
# Feature Selection
# -------------------------
features = [
    "GrLivArea",
    "BedroomAbvGr",
    "FullBath",
    "OverallQual",
    "YearBuilt"
]

X = train[features].fillna(0).values
y = train["SalePrice"].values

X = jnp.array(X)
y = jnp.array(y)

# -------------------------
# NORMALIZATION
# -------------------------
X_mean = jnp.mean(X, axis=0)
X_std = jnp.std(X, axis=0) + 1e-8
X = (X - X_mean) / X_std

y_mean = jnp.mean(y)
y_std = jnp.std(y) + 1e-8
y = (y - y_mean) / y_std

# -------------------------
# Add bias column
# -------------------------
X = jnp.c_[jnp.ones(X.shape[0]), X]

# -------------------------
# Model
# -------------------------
def predict(w, X):
    return jnp.dot(X, w)

def loss_fn(w, X, y):
    preds = predict(w, X)
    return jnp.mean((preds - y) ** 2)

# -------------------------
# Grad
# -------------------------
grad_fn = grad(loss_fn)

# -------------------------
# Update Step
# -------------------------
@jit
def update(w, X, y, lr):
    return w - lr * grad_fn(w, X, y)

# -------------------------
# Training Loop
# -------------------------
@jit
def train_loop(w, X, y, lr, steps):
    def body(i, w):
        return update(w, X, y, lr)
    return fori_loop(0, steps, body, w)

# -------------------------
# Initialize Weights
# -------------------------
key = jax.random.PRNGKey(0)
w = jax.random.normal(key, (X.shape[1],))

# -------------------------
# TIMING START
# -------------------------
start_time = time.time()

# -------------------------
# Train Model
# -------------------------
w = train_loop(w, X, y, lr=0.005, steps=10000)

# -------------------------
# TIMING END
# -------------------------
end_time = time.time()
train_time = end_time - start_time

# -------------------------
# Predictions
# -------------------------
preds = predict(w, X)

# -------------------------
# Convert back to original scale
# -------------------------
preds = preds * y_std + y_mean
y_original = y * y_std + y_mean

# -------------------------
# Evaluation
# -------------------------
mse = jnp.mean((preds - y_original) ** 2)
rmse = jnp.sqrt(mse)

# -------------------------
# OUTPUT
# -------------------------
print("MSE:", mse)
print("RMSE:", rmse)
print("Training Time:", train_time, "seconds")
print("First 5 Predictions:", preds[:5])

# House Prices Prediction (NumPy + Pandas Only)

In [ ]:
import numpy as np
import pandas as pd
import time

# -------------------------
# Load Dataset
# -------------------------
path = "/kaggle/input/competitions/house-prices-advanced-regression-techniques/"
train = pd.read_csv(path + "train.csv")

# -------------------------
# Feature Selection
# -------------------------
features = [
    "GrLivArea",
    "BedroomAbvGr",
    "FullBath",
    "OverallQual",
    "YearBuilt"
]

X = train[features].fillna(0).values.astype(np.float64)
y = train["SalePrice"].values.astype(np.float64)

# -------------------------
# NORMALIZATION
# -------------------------
X_mean = np.mean(X, axis=0)
X_std = np.std(X, axis=0) + 1e-8
X = (X - X_mean) / X_std

y_mean = np.mean(y)
y_std = np.std(y) + 1e-8
y = (y - y_mean) / y_std

# -------------------------
# Add bias column
# -------------------------
X = np.c_[np.ones(X.shape[0]), X]

# -------------------------
# Model
# -------------------------
def predict(w, X):
    return np.dot(X, w)

def loss_fn(w, X, y):
    preds = predict(w, X)
    return np.mean((preds - y) ** 2)

# -------------------------
# Manual Gradient (NO AUTODIFF)
# -------------------------
def compute_grad(w, X, y):
    preds = predict(w, X)
    error = preds - y
    grad = (2 / X.shape[0]) * np.dot(X.T, error)
    return grad

# -------------------------
# Training Loop
# -------------------------
def train(X, y, lr=0.005, steps=10000):
    start_time = time.time()

    w = np.random.randn(X.shape[1])

    for i in range(steps):
        grad = compute_grad(w, X, y)
        w = w - lr * grad

        if i % 1000 == 0:
            loss = loss_fn(w, X, y)
            print(f"Step {i}, Loss: {loss}")

    end_time = time.time()

    return w, end_time - start_time

# -------------------------
# Train Model
# -------------------------
w, train_time = train(X, y, lr=0.005, steps=10000)

# -------------------------
# Predictions
# -------------------------
preds = predict(w, X)

# -------------------------
# Convert back to original scale
# -------------------------
preds = preds * y_std + y_mean
y_original = y * y_std + y_mean

# -------------------------
# Evaluation
# -------------------------
mse = np.mean((preds - y_original) ** 2)
rmse = np.sqrt(mse)

print("\n-------------------------")
print("NUMPY RESULTS")
print("-------------------------")
print("MSE:", mse)
print("RMSE:", rmse)
print("Training Time:", train_time, "seconds")
print("First 5 Predictions:", preds[:5])

# Store in csv and submit for ranking

In [ ]:
# -------------------------
# LOAD TEST DATA
# -------------------------
test = pd.read_csv(path + "test.csv")

# Keep Ids for submission
test_ids = test["Id"]

# -------------------------
# SAME FEATURE SELECTION
# -------------------------
X_test = test[features].fillna(0).values
X_test = jnp.array(X_test)

# -------------------------
# APPLY SAME NORMALIZATION
# (IMPORTANT ⚠️ use TRAIN mean/std)
# -------------------------
X_test = (X_test - X_mean) / X_std

# -------------------------
# ADD BIAS COLUMN
# -------------------------
X_test = jnp.c_[jnp.ones(X_test.shape[0]), X_test]

# -------------------------
# PREDICT
# -------------------------
test_preds = predict(w, X_test)

# -------------------------
# CONVERT BACK TO ORIGINAL SCALE
# -------------------------
test_preds = test_preds * y_std + y_mean

# Convert to numpy (for pandas)
test_preds = jnp.array(test_preds)

# -------------------------
# CREATE SUBMISSION FILE
# -------------------------
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": test_preds
})

# -------------------------
# SAVE CSV
# -------------------------
submission.to_csv("submission.csv", index=False)

print("submission.csv file created successfully!")
print(submission.head())

# Download csv and submit

In [ ]:
from IPython.display import FileLink

# Save CSV (already done, but keeping for safety)
submission.to_csv("submission.csv", index=False)

print("File ready for download 👇")

# Create clickable download link
FileLink("submission.csv")